### Ingestion del archivo "genre.csv"

In [0]:
dbutils.widgets.text("p_environment", "")
v_environment = dbutils.widgets.get("p_environment")

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-16")
v_file_date = dbutils.widgets.get("p_file_date")

#### Paso 1 - Leer el archivo CSV usando "DataFrameReader" de Spark 

In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
from pyspark.sql.types import StructField, StructType, IntegerType, StringType

In [0]:
genres_schema = StructType(fields=[
    StructField("genreId", IntegerType(), False),
    StructField("genreName", StringType(), True)
])

In [0]:
genres_df = spark.read \
    .option("header", True) \
    .schema(genres_schema) \
    .csv(f"{bronze_folder_path}/{v_file_date}/genre.csv")

#### Paso 2 - Cambiar el nombre de las columnas según lo "requerido"

In [0]:
genres_renamed_df = genres_df.withColumnsRenamed({"genreId": "genre_id", "genreName": "genre_name"})

#### Paso 3 - Agregar la columna "ingestion_date" y "enviroment" al DataFrame

In [0]:
from pyspark.sql.functions import lit

In [0]:
genres_final_df = add_ingestion_date(genres_renamed_df) \
    .withColumn("environment", lit(v_environment)) \
    .withColumn("file_date", lit(v_file_date))

#### Paso 4 - Escribir datos en el DataLake en formato "Delta"

In [0]:
genres_final_df.write.mode("overwrite").format("delta").saveAsTable("movie_silver.genres")

In [0]:
%sql
SELECT * FROM movie_silver.genres;

genre_id,genre_name,ingestion_date,environment,file_date
12,Adventure,2026-09-21T16:16:24.994475Z,production,2024-12-16
14,Fantasy,2026-09-21T16:16:24.994475Z,production,2024-12-16
16,Animation,2026-09-21T16:16:24.994475Z,production,2024-12-16
18,Drama,2026-09-21T16:16:24.994475Z,production,2024-12-16
27,Horror,2026-09-21T16:16:24.994475Z,production,2024-12-16
28,Action,2026-09-21T16:16:24.994475Z,production,2024-12-16
35,Comedy,2026-09-21T16:16:24.994475Z,production,2024-12-16
36,History,2026-09-21T16:16:24.994475Z,production,2024-12-16
37,Western,2026-09-21T16:16:24.994475Z,production,2024-12-16
53,Thriller,2026-09-21T16:16:24.994475Z,production,2024-12-16


In [0]:
dbutils.notebook.exit("Success")